# Kaggle: regenerate `baseline_gen.jsonl` for the LLM-judge sample only

The original full 803-row `baseline_gen.jsonl` download was overwritten locally before it got copied into the repo (only the aggregate `summary.csv` numbers survived, already recorded). Rather than redo the full 803-row baseline eval, this regenerates **just the 150-row LLM-judge common subsample** (`data/raw/sft_test_judge_sample.jsonl`) -- `src/eval/llm_judge.py`'s `--ids-file` filter would only use those 150 ids out of a full run anyway, so a full re-run would waste GPU-hours reproducing rows that are never going to be judged.

Zero-shot, no adapter -- same as the original baseline run, just a smaller input file.

**Before running:**
1. Zip `src/` (contents) + `config.yaml` as `src.zip`.
2. Upload that zip plus `data/raw/sft_test_judge_sample.jsonl` as a Kaggle Dataset.
3. Single T4 accelerator, Internet on.
4. Run all cells.

**Output:** `/kaggle/working/results/baseline_gen.jsonl` (150 rows) -- download and place at `results/baseline_gen.jsonl` locally (this intentionally overwrites nothing, since no full baseline_gen.jsonl currently exists locally -- it's the file `kaggle_llm_judge.ipynb` needs as one of its 5 inputs).

In [ ]:
!pip install -q -U "trl==1.10.0" "peft==0.20.0" "transformers==5.15.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" pyyaml
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import os, sys, zipfile

def find_repo(root="/kaggle/input"):
    repo_dir = None
    config_path = None
    zip_path = None
    for r, dirs, files in os.walk(root):
        if "build_irac.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                repo_dir = os.path.dirname(src_dir)
        if config_path is None and "config.yaml" in files:
            config_path = os.path.join(r, "config.yaml")
        if "src.zip" in files:
            zip_path = os.path.join(r, "src.zip")
    return repo_dir, config_path, zip_path

repo_dir, config_path, zip_path = find_repo()

if repo_dir is None and zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    repo_dir, config_path, _ = find_repo("/kaggle/working/repo")
    print("Extracted", zip_path)

if repo_dir is None or config_path is None:
    raise FileNotFoundError(
        f"repo_dir={repo_dir}, config_path={config_path} -- could not find both "
        "src/data/build_irac.py and config.yaml under /kaggle/input (in any "
        "nesting). Check the dataset is attached."
    )

REPO_DIR = repo_dir
sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)

import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)

MODEL_ID = cfg["model"]["candidates"][cfg["model"]["active"]]["hf_id"]
print("Active model:", MODEL_ID)

from src.eval.generate import run as generate_run

In [ ]:
def find_data_file(name, root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f"{name} not found under {root} -- check it was included in the uploaded dataset.")

SAMPLE_FILE = find_data_file("sft_test_judge_sample.jsonl")
print(SAMPLE_FILE)

RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
BASELINE_GEN = os.path.join(RESULTS_DIR, "baseline_gen.jsonl")

generate_run(
    input_path=SAMPLE_FILE,
    output_path=BASELINE_GEN,
    model_id=MODEL_ID,
    adapter_path=None,
    load_in_4bit=False,
    batch_size=16,
    max_new_tokens=350,
    limit=None,
)

In [ ]:
import json
with open(BASELINE_GEN, encoding="utf-8") as f:
    rows = [json.loads(l) for l in f]
print(f"{len(rows)} rows generated")
print(rows[0]["raw_response"][:300])